# Suspicious Login Detection

## 0. Introduction

In today's increasingly digital world, network security has become a critical concern as systems are constantly exposed to potential cyber threats. Detecting malicious activities such as unauthorized access, denial-of-service attacks, and other intrusion attempts is essential for maintaining the integrity and availability of systems.

This project aims to develop a machine learning model to classify network connections as either normal or malicious based on various connection features. Using the NSL-KDD dataset, which contains simulated network traffic data, the objective is to identify patterns that distinguish legitimate activities from potential attacks.

The problem is formulated as a binary classification task, where normal connections are labeled as 0 and all forms of attacks are labeled as 1. A Logistic Regression model is employed as the primary algorithm, and its performance is evaluated using metrics such as accuracy, precision, recall, F1-score, and ROC-AUC. A dummy baseline model is also included for comparison to assess the effectiveness of the trained model.

By applying data preprocessing, feature engineering, and model evaluation techniques, this project demonstrates how machine learning can be used as a tool for basic intrusion detection in cybersecurity.

## 1. Setup, Importing Libraries, loading Data, Preprocessing Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#load and read data

df = pd.read_csv('nsl_kdd.csv', header=None, index_col=0)

df.head()

,1,2,3,4,5,6,7,8,9,10,...,34,35,36,37,38,39,40,41,42,43
0,,,,,,,,,,,,,,,,,,,,,
NaN,0,1,2,3,4,5,6,7,8,9,...,33.00,34.00,35.00,36.00,37.00,38.00,39.00,40.00,41,42
0.0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1.0,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2.0,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3.0,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21


In [3]:
df.shape

(125974, 43)

In [4]:
# add column names

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df.columns = columns

In [5]:
# understand the target

df['label'].value_counts()

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
41                     1
Name: count, dtype: int64

In [6]:
# simplify to binary classification by converting normal to 0 and attack to 1

df['target'] = df['label'].apply(lambda x: 0 if x=='normal' else 1)

In [7]:
# drop unnecessary columns from dataset

df = df.drop(['label', 'difficulty'], axis=1)

In [8]:
# identify categorical features

df.dtypes

duration                         int64
protocol_type                   object
service                         object
flag                            object
src_bytes                        int64
dst_bytes                        int64
land                             int64
wrong_fragment                   int64
urgent                           int64
hot                              int64
num_failed_logins                int64
logged_in                        int64
num_compromised                  int64
root_shell                       int64
su_attempted                     int64
num_root                         int64
num_file_creations               int64
num_shells                       int64
num_access_files                 int64
num_outbound_cmds                int64
is_host_login                    int64
is_guest_login                   int64
count                            int64
srv_count                        int64
serror_rate                    float64
srv_serror_rate          

## 2. One-Hot Encoding Categorical Features

In [9]:
# one-hot encode categorical features

df_encoded = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'], drop_first=True)

df_encoded.shape

(125974, 123)

## 3. Define x and y

In [10]:
x = df_encoded.drop('target', axis=1)
y = df_encoded['target']

## 4. Train-Test Split

In [11]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, stratify=y, random_state=42)

## 5. Feature Scaling

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## 6. Training Logistic Regression Model

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(x_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## 7. Predictions

In [14]:
y_pred = model.predict(x_test_scaled)
y_proba = model.predict_proba(x_test_scaled)[:,1]

## 8. Evaluate Logistic Regression Model

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print('Accuracy: ', accuracy)
print('Precision: ', precision)
print('Recall: ', recall)
print('F1-score: ', f1)
print('ROC-AUC: ', roc_auc)

Accuracy:  0.9727991956182361
Precision:  0.9774561808118081
Recall:  0.9637862421830585
F1-score:  0.9705730806663995
ROC-AUC:  0.9959570042859743


## 9. Dummy Baseline

In [16]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(x_train_scaled, y_train)

y_pred_d = dummy.predict(x_test_scaled)
y_proba_d = dummy.predict_proba(x_test_scaled)[:,1]

accuracy_d = accuracy_score(y_test, y_pred_d)
precision_d = precision_score(y_test, y_pred_d, zero_division=0)
recall_d = recall_score(y_test, y_pred_d)
f1_d = f1_score(y_test, y_pred_d)
roc_auc_d = roc_auc_score(y_test, y_proba_d)

print('Dummy Accuracy: ', accuracy_d)
print('Dummy Precision: ', precision_d)
print('Dummy Recall: ', recall_d)
print('Dummy F1-score: ', f1_d)
print('Dummy ROC-AUC: ', roc_auc_d)

Dummy Accuracy:  0.534569893895695
Dummy Precision:  0.0
Dummy Recall:  0.0
Dummy F1-score:  0.0
Dummy ROC-AUC:  0.5


## 10. Tabulate Results

In [17]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Dummy Baseline'],
    'Accuracy': [accuracy, accuracy_d],
    'Precision': [precision, precision_d],
    'Recall': [recall, recall_d],
    'F1-Score': [f1, f1_d ],
    'ROC-AUC Score': [roc_auc, roc_auc_d]
}).set_index('Model')

print(results)

                     Accuracy  Precision    Recall  F1-Score  ROC-AUC Score
Model                                                                      
Logistic Regression  0.972799   0.977456  0.963786  0.970573       0.995957
Dummy Baseline       0.534570   0.000000  0.000000  0.000000       0.500000


## 11. Checking Validity of Results

In [18]:
# check class distribution in train and test datasets

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


target
0    0.534582
1    0.465418
Name: proportion, dtype: float64
target
0    0.53457
1    0.46543
Name: proportion, dtype: float64


In [19]:
# check if any feature perfectly predicts target

for col in x_train.columns:
    if x_train[col].nunique() == 2:
        print(col, x_train[col].value_counts())

num_outbound_cmds num_outbound_cmds
0     88180
19        1
Name: count, dtype: int64
protocol_type_icmp protocol_type_icmp
False    82414
True      5767
Name: count, dtype: int64
protocol_type_tcp protocol_type_tcp
True     71889
False    16292
Name: count, dtype: int64
protocol_type_udp protocol_type_udp
False    77657
True     10524
Name: count, dtype: int64
service_IRC service_IRC
False    88056
True       125
Name: count, dtype: int64
service_X11 service_X11
False    88128
True        53
Name: count, dtype: int64
service_Z39_50 service_Z39_50
False    87590
True       591
Name: count, dtype: int64
service_aol service_aol
False    88180
True         1
Name: count, dtype: int64
service_auth service_auth
False    87504
True       677
Name: count, dtype: int64
service_bgp service_bgp
False    87674
True       507
Name: count, dtype: int64
service_courier service_courier
False    87678
True       503
Name: count, dtype: int64
service_csnet_ns service_csnet_ns
False    87800
True       

In [20]:
# detect "almost perfect" predictors

threshold = 0.99  # 99% match

suspicious_features = []

for col in x_train.columns:
    match_rate = (x_train[col] == y_train).mean()
    inverse_match_rate = (x_train[col] != y_train).mean()

    if match_rate > threshold or inverse_match_rate > threshold:
        suspicious_features.append((col, match_rate, inverse_match_rate))

if suspicious_features:
    print('Suspicious features (possible leakage): ')
    for feat in suspicious_features:
        print(feat)
else:
    print('No suspicious near-perfect predictors detected.')

No suspicious near-perfect predictors detected.


## 12. Conducting Cross-Validation

In [21]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    x_train_scaled,
    y_train,
    cv=skf,
    scoring='roc_auc'
)

print('Fold Scores: ', cv_scores)
print('Mean AUC: ', cv_scores.mean())

Fold Scores:  [0.99625374 0.99625725 0.99695184 0.99568256 0.99736693]
Mean AUC:  0.9965024623363068


## 13. Conclusion

The model achieved a mean ROC-AUC of 0.9965 across 5-fold cross-validation, with minimal variance between folds. This indicates that the model's performance is stable and not dependent on a particular train-test split.

The results demonstrate that Logistic Regression can effectively distinguish normal network connections from malicious connections within the NSL-KDD dataset. The high ROC-AUC suggests that the engineered network-traffic features contain strong signals that can be used for binary intrusion classification.

However, the model's performance on the NSL-KDD dataset should not be interpreted as equivalent to performance in a real-world network environment. NSL-KDD is a simulated benchmark dataset, and real network traffic may contain different patterns and evolving attack patterns.

Overall, this project demonstrates how supervised machine learning can be applied as a basic intrusion-detection approach and provides a foundation for further exploration of cybersecurity analytics and anomaly detection.